# VITRA Inference-Only Ablation

This notebook runs low-cost component ablations on the EPIC clean test split without retraining. It keeps the same checkpoint and sampler split, then changes only inference inputs such as state and FOV.

## Plan

1. Run smoke ablations on a small clean EPIC subset.
2. Inspect which ablations produce a visible loss gap.
3. Run full EPIC test-split eval only for the useful settings.

The clean split parameters match the previous main eval: `CUTOFF=128000`, `seen_sampler_steps=128000`, and `exclude_seen_ids=True`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap
import pandas as pd

RESOURCE_ROOT = Path("/kaggle/input/notebooks/ldthanh/prepare-vitra-resources")
RESOURCE_REPO = RESOURCE_ROOT / "VITRA"
RESOURCE_WHEELS = RESOURCE_ROOT / "wheels"
WORK_REPO = Path("/kaggle/working/VITRA")

if Path("/kaggle/working").exists():
    if not WORK_REPO.exists():
        assert RESOURCE_REPO.exists(), f"Missing prepared VITRA repo: {RESOURCE_REPO}. Attach prepare-vitra-resources as a Kaggle input."
        shutil.copytree(RESOURCE_REPO, WORK_REPO)
    os.chdir(WORK_REPO)
    repo = WORK_REPO
else:
    repo = Path.cwd()
    assert (repo / "scripts" / "evaluate_pretrained_loss.py").exists(), "Run locally from the repo root."

print("Repo:", repo)
print("Resource root exists:", RESOURCE_ROOT.exists())
print("Offline wheels exists:", RESOURCE_WHEELS.exists())


In [ ]:
%%bash
set -e
if [ -d /kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels ]; then
  uv pip install scipy     "torch"     "torchvision"     "torchaudio"     "PyYAML==6.0.2"     "hydra-colorlog>=1.2.0"     "hydra-core>=1.1.1"     "deepspeed==0.16.5"     "tensorboard>=2.13.0"     "tensorboardX>=2.6.2"     "tqdm>=4.65.0"     "transformers==4.47.1"     "diffusers>=0.31.0"     "wandb>=0.19.0"     "numpy<2.0"     "sentence-transformers==2.2.2"     "open_clip_torch==2.20.0"     "datasets==2.12.0"     "draccus==0.8.0"     "einops"     "huggingface_hub"     "json-numpy"     "jsonlines"     "matplotlib"     "peft==0.11.1"     "protobuf"     "rich"     "sentencepiece==0.1.99"     "timm==0.9.10"     "tokenizers>=0.21"     "decord"     "scikit-image"     "brotli"     "imageio-ffmpeg"     "imageio"     "ffmpeg-python"     "opencv-python"     "pympler"     "ultralytics"     "pytorch-lightning"     "yacs"     "utils3d"     --no-index --find-links "/kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels"
else
  echo "[INFO] Offline wheels not found; assuming dependencies are already installed."
fi


In [ ]:
from pathlib import Path

EVAL_SCRIPT = '"""\nEvaluate a VITRA weights.pt checkpoint on a deterministic held-out sampler slice.\n\nThis script intentionally mirrors scripts/train.py for model construction and data\nmaterialization, but runs forward-only loss evaluation from a chosen sampler step.\nUse it to evaluate samples that were not consumed during a partial epoch training run.\n"""\n\nimport argparse\nimport copy\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Dict\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))\n\nfrom vitra.datasets.materialize import get_vla_dataset_and_collator\nfrom vitra.datasets.data_mixture import HAND_MIXTURES\nfrom vitra.models.vla_builder import build_vla, load_vla_checkpoint\nfrom vitra.utils import set_global_seed, setup_seed\nfrom vitra.utils.config_utils import load_config\nfrom vitra.utils.overwatch import initialize_overwatch\n\n\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\n\noverwatch = initialize_overwatch(__name__)\n\n\nclass FixedBatchSampler:\n    def __init__(self, batches):\n        self.batches = batches\n\n    def __iter__(self):\n        yield from self.batches\n\n    def __len__(self):\n        return len(self.batches)\n\n\ndef move_to_device(value: Any, device: torch.device) -> Any:\n    if torch.is_tensor(value):\n        return value.to(device, non_blocking=True)\n    if isinstance(value, dict):\n        return {k: move_to_device(v, device) for k, v in value.items()}\n    if isinstance(value, list):\n        return [move_to_device(v, device) for v in value]\n    if isinstance(value, tuple):\n        return tuple(move_to_device(v, device) for v in value)\n    return value\n\n\ndef tensor_to_float(value: Any) -> float:\n    if torch.is_tensor(value):\n        return float(value.detach().float().mean().cpu().item())\n    return float(value)\n\n\ndef posix_to_str(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {k: posix_to_str(v) for k, v in value.items()}\n    if isinstance(value, list):\n        return [posix_to_str(v) for v in value]\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef resolve_weights_path(path: str) -> str:\n    weights_path = Path(path)\n    if weights_path.is_dir():\n        weights_path = weights_path / "weights.pt"\n    if not weights_path.exists():\n        raise FileNotFoundError(f"Checkpoint weights not found: {weights_path}")\n    return str(weights_path)\n\n\ndef cyclic_shift_batch(value: torch.Tensor) -> torch.Tensor:\n    if value.shape[0] <= 1:\n        return value\n    return torch.roll(value, shifts=1, dims=0)\n\n\ndef get_local_rank() -> int:\n    local_rank = getattr(overwatch, "local_rank", None)\n    if callable(local_rank):\n        return int(local_rank())\n    return int(os.environ.get("LOCAL_RANK", 0))\n\n\ndef apply_inference_ablation(batch: Dict[str, Any], args: argparse.Namespace) -> Dict[str, Any]:\n    """Apply input-only ablations for forward-loss evaluation."""\n    state_mode = args.ablate_state\n    if state_mode == "zero_state":\n        batch["current_state"] = torch.zeros_like(batch["current_state"])\n    elif state_mode == "no_state":\n        batch["current_state"] = torch.zeros_like(batch["current_state"])\n        batch["current_state_mask"] = torch.zeros_like(batch["current_state_mask"], dtype=torch.bool)\n    elif state_mode == "shuffle_state":\n        batch["current_state"] = cyclic_shift_batch(batch["current_state"])\n        batch["current_state_mask"] = cyclic_shift_batch(batch["current_state_mask"])\n\n    fov_mode = args.ablate_fov\n    if fov_mode == "zero_fov":\n        batch["fov"] = torch.zeros_like(batch["fov"])\n    elif fov_mode == "shuffle_fov":\n        batch["fov"] = cyclic_shift_batch(batch["fov"])\n\n    return batch\n\n\ndef update_configs(configs: Dict[str, Any], args: argparse.Namespace) -> Dict[str, Any]:\n    configs = copy.deepcopy(configs)\n\n    if args.seed is not None:\n        configs["seed"] = args.seed\n    if args.data_mix is not None:\n        configs["train_dataset"]["data_mix"] = args.data_mix\n    if args.batch_size is not None:\n        configs["batch_size"] = args.batch_size\n    if args.total_batch_size is not None:\n        configs["total_batch_size"] = args.total_batch_size\n    if args.fwd_pred_next_n is not None:\n        configs["fwd_pred_next_n"] = args.fwd_pred_next_n\n    if args.repeated_diffusion_steps is not None:\n        configs["repeated_diffusion_steps"] = args.repeated_diffusion_steps\n    if args.use_bf16:\n        configs["use_bf16"] = True\n\n    configs["output_root"] = Path(configs["output_root"])\n    configs["log_root"] = Path(configs["log_root"])\n    configs["cache_root"] = Path(configs["cache_root"]) / configs["model"]\n    if args.weights != "__dry_run_placeholder__":\n        configs["model_load_path"] = resolve_weights_path(args.weights)\n    else:\n        configs["model_load_path"] = None  # dry_run: no checkpoint needed\n\n    if not args.keep_train_augmentation:\n        train_dataset = configs["train_dataset"]\n        train_dataset["augmentation"] = False\n        train_dataset["state_mask_prob"] = 0.0\n        train_dataset["set_none_ratio"] = 0.0\n\n    return configs\n\n\ndef collect_sampler_ids(batch_sampler, epoch: int, start_step: int, num_steps: int) -> set:\n    batch_sampler.set_epoch(epoch, start_step)\n    ids = set()\n    for step_idx, batch in zip(range(num_steps), batch_sampler):\n        ids.update(tuple(index) for index in batch)\n    return ids\n\n\ndef count_available_samples(\n    batch_sampler,\n    epoch: int,\n    cutoff_step: int,\n    num_datasets: int,\n    seen_ids: set = None,\n    exclude_seen: bool = True,\n) -> Dict[int, int]:\n    """Count unique unseen samples available per dataset from cutoff_step to end-of-epoch.\n\n    Returns a dict mapping dataset_id -> count of qualifying unique samples.\n    This runs without loading the model and is fast enough to sweep across checkpoints.\n    """\n    batch_sampler.set_epoch(epoch, cutoff_step)\n    seen_ids = seen_ids or set()\n    per_dataset_seen: Dict[int, set] = {i: set() for i in range(num_datasets)}\n    counts: Dict[int, int] = {i: 0 for i in range(num_datasets)}\n\n    for mixed_batch in batch_sampler:\n        for index in mixed_batch:\n            index = tuple(index)\n            dataset_id = index[0]\n            if dataset_id not in counts:\n                continue\n            if exclude_seen and index in seen_ids:\n                continue\n            if index in per_dataset_seen[dataset_id]:\n                continue\n            per_dataset_seen[dataset_id].add(index)\n            counts[dataset_id] += 1\n\n    return counts\n\n\ndef get_dataset_names(vla_dataset) -> list:\n    names = []\n    for dataset in vla_dataset.datasets:\n        names.append(getattr(dataset, "dataset_name", f"dataset_{len(names)}"))\n    return names\n\n\ndef get_config_dataset_names(configs: Dict[str, Any]) -> list:\n    data_mix = configs["train_dataset"]["data_mix"]\n    if data_mix in HAND_MIXTURES:\n        return [dataset_name for dataset_name, _ in HAND_MIXTURES[data_mix]]\n    return [data_mix]\n\n\ndef collect_dataset_batches(\n    batch_sampler,\n    epoch: int,\n    start_step: int,\n    dataset_id: int,\n    num_batches: int,\n    batch_size: int,\n    exclude_ids: set = None,\n    unique: bool = True,\n) -> list:\n    """Filter the mixed sampler stream to fixed-size batches from one dataset id."""\n    batch_sampler.set_epoch(epoch, start_step)\n    batches = []\n    current_batch = []\n    exclude_ids = exclude_ids or set()\n    used_ids = set()\n\n    for mixed_batch in batch_sampler:\n        for index in mixed_batch:\n            index = tuple(index)\n            if index[0] != dataset_id:\n                continue\n            if index in exclude_ids:\n                continue\n            if unique and index in used_ids:\n                continue\n\n            current_batch.append(index)\n            used_ids.add(index)\n            if len(current_batch) == batch_size:\n                batches.append(current_batch)\n                current_batch = []\n                if len(batches) == num_batches:\n                    return batches\n\n    raise RuntimeError(\n        f"Could only collect {len(batches)} batches for dataset_id={dataset_id}; "\n        f"requested {num_batches}. Try a smaller --eval_batches, earlier --eval_sampler_step, "\n        f"or disable --unique_per_dataset_eval."\n    )\n\n\ndef make_dataset_and_sampler(configs: Dict[str, Any], processor, args: argparse.Namespace):\n    batch_size = configs["batch_size"]\n    train_dataset = configs["train_dataset"]\n\n    vla_dataset, collator, batch_sampler = get_vla_dataset_and_collator(\n        train_dataset["data_root_dir"],\n        train_dataset["data_mix"],\n        augmentation=train_dataset["augmentation"],\n        shard_num=overwatch.world_size(),\n        shard_index=overwatch.rank(),\n        seed=configs["seed"],\n        future_action_window_size=configs["fwd_pred_next_n"] - 1,\n        processor=processor,\n        batch_size=batch_size,\n        normalization=train_dataset.get("normalization", True),\n        flip_augmentation=train_dataset.get("flip_augmentation", 1.0),\n        set_none_ratio=train_dataset.get("set_none_ratio", 0.0),\n        action_type=train_dataset.get("action_type", "angle"),\n        use_rel=train_dataset.get("use_rel", False),\n        rel_mode=train_dataset.get("rel_mode", "step"),\n        clip_len=train_dataset.get("clip_len", None),\n        state_mask_prob=train_dataset.get("state_mask_prob", 0.0),\n        target_image_height=train_dataset.get("target_image_height", 224),\n    )\n\n    setup_seed(configs["seed"], rank=overwatch.rank())\n    return vla_dataset, collator, batch_sampler\n\n\ndef make_dataloader(vla_dataset, collator, batch_sampler, args: argparse.Namespace, configs: Dict[str, Any]):\n    train_dataset = configs["train_dataset"]\n    num_workers = args.num_workers\n    if num_workers is None:\n        num_workers = train_dataset["num_workers"]\n    prefetch_factor = args.prefetch_factor\n    if prefetch_factor is None:\n        prefetch_factor = train_dataset["prefetch_factor"]\n    if num_workers == 0 or prefetch_factor == 0:\n        prefetch_factor = None\n\n    dataloader = DataLoader(\n        vla_dataset,\n        batch_sampler=batch_sampler,\n        collate_fn=collator,\n        num_workers=num_workers,\n        prefetch_factor=prefetch_factor,\n        worker_init_fn=set_global_seed(configs["seed"], get_worker_init_fn=True),\n        persistent_workers=num_workers > 0,\n        pin_memory=num_workers > 0,\n    )\n    return dataloader, batch_sampler\n\n\n@torch.no_grad()\ndef evaluate(configs: Dict[str, Any], args: argparse.Namespace, dataset_name: str = None) -> Dict[str, Any]:\n    device = torch.device(f"cuda:{get_local_rank()}" if torch.cuda.is_available() else "cpu")\n    if torch.cuda.is_available():\n        torch.cuda.set_device(device)\n        torch.cuda.empty_cache()\n\n    model = build_vla(configs=configs)\n    model = load_vla_checkpoint(model, configs["model_load_path"])\n    model.model.use_bf16 = configs["use_bf16"]\n    model.use_bf16 = configs["use_bf16"]\n    model = model.to(device).eval()\n\n    vla_dataset, collator, batch_sampler = make_dataset_and_sampler(configs, model.processor, args)\n    dataset_names = get_dataset_names(vla_dataset)\n\n    overlap_count = None\n    seen_ids = None\n    if args.seen_sampler_steps is not None:\n        seen_ids = collect_sampler_ids(batch_sampler, args.seen_epoch, 0, args.seen_sampler_steps)\n\n    if dataset_name is None:\n        if seen_ids is not None:\n            eval_ids = collect_sampler_ids(batch_sampler, args.eval_epoch, args.eval_sampler_step, args.eval_batches)\n            overlap = seen_ids & eval_ids\n            overlap_count = len(overlap)\n            if overlap_count > 0 and not args.allow_seen_overlap:\n                raise RuntimeError(\n                    f"Evaluation sampler slice overlaps the seen sampler slice by {overlap_count} sample ids. "\n                    f"This can happen with weighted oversampling. Increase --eval_sampler_step or pass "\n                    f"--allow_seen_overlap for a weighted reference-set comparison."\n                )\n        batch_sampler.set_epoch(args.eval_epoch, args.eval_sampler_step)\n        eval_batch_sampler = batch_sampler\n        eval_dataset_id = None\n    else:\n        if dataset_name not in dataset_names:\n            raise ValueError(f"Unknown dataset {dataset_name!r}. Available datasets: {dataset_names}")\n        eval_dataset_id = dataset_names.index(dataset_name)\n        fixed_batches = collect_dataset_batches(\n            batch_sampler,\n            args.eval_epoch,\n            args.eval_sampler_step,\n            eval_dataset_id,\n            args.eval_batches,\n            configs["batch_size"],\n            exclude_ids=seen_ids if args.exclude_seen_ids else None,\n            unique=args.unique_per_dataset_eval,\n        )\n        eval_batch_sampler = FixedBatchSampler(fixed_batches)\n\n        if args.seen_sampler_steps is not None:\n            eval_ids = {index for batch in fixed_batches for index in batch}\n            overlap_count = len(seen_ids & eval_ids)\n            if overlap_count > 0:\n                raise RuntimeError(\n                    f"Per-dataset evaluation for {dataset_name} overlaps the seen sampler slice by "\n                    f"{overlap_count} sample ids. Increase --eval_sampler_step."\n                )\n\n    dataloader, _ = make_dataloader(vla_dataset, collator, eval_batch_sampler, args, configs)\n\n    output_jsonl = Path(args.output_jsonl)\n    if dataset_name is not None:\n        output_jsonl = output_jsonl.with_name(f"{output_jsonl.stem}.{dataset_name}{output_jsonl.suffix}")\n    output_jsonl.parent.mkdir(parents=True, exist_ok=True)\n\n    component_sums: Dict[str, float] = {}\n    component_values: Dict[str, list] = {}\n    num_batches = 0\n    num_examples = 0\n\n    progress = tqdm(\n        zip(range(args.eval_batches), dataloader),\n        total=args.eval_batches,\n        disable=not overwatch.is_rank_zero(),\n        desc="evaluating",\n    )\n    with open(output_jsonl, "w") as f:\n        for local_step, batch in progress:\n            batch = move_to_device(batch, device)\n            batch = apply_inference_ablation(batch, args)\n            prediction = model.forward(\n                batch["pixel_values"],\n                batch["input_ids"],\n                attention_mask=batch["attention_mask"],\n                action_labels=batch["actions"],\n                action_masks=batch["action_masks"],\n                current_state_mask=batch["current_state_mask"],\n                current_state=batch["current_state"],\n                data_source=["action"],\n                fov=batch["fov"],\n            )\n\n            record = {\n                "eval_batch": local_step,\n                "sampler_epoch": args.eval_epoch,\n                "sampler_step": args.eval_sampler_step + local_step if dataset_name is None else None,\n                "source_start_sampler_step": args.eval_sampler_step,\n                "batch_size": int(batch["input_ids"].shape[0]),\n                "dataset": dataset_name or "mixed",\n                "ablate_state": args.ablate_state,\n                "ablate_fov": args.ablate_fov,\n                "repeated_diffusion_steps": configs.get("repeated_diffusion_steps"),\n            }\n            for key, value in prediction.items():\n                record[key] = tensor_to_float(value)\n                component_sums[key] = component_sums.get(key, 0.0) + record[key]\n                component_values.setdefault(key, []).append(record[key])\n\n            num_batches += 1\n            num_examples += record["batch_size"]\n            progress.set_postfix(loss=record.get("loss"))\n            f.write(json.dumps(record) + "\\n")\n\n    summary = {\n        "checkpoint": configs["model_load_path"],\n        "dataset": dataset_name or "mixed",\n        "dataset_id": eval_dataset_id,\n        "dataset_names": dataset_names,\n        "config": posix_to_str(configs),\n        "eval_epoch": args.eval_epoch,\n        "eval_sampler_step": args.eval_sampler_step,\n        "eval_batches": num_batches,\n        "eval_examples": num_examples,\n        "seen_epoch": args.seen_epoch,\n        "seen_sampler_steps": args.seen_sampler_steps,\n        "seen_eval_overlap": overlap_count,\n        "exclude_seen_ids": args.exclude_seen_ids if dataset_name is not None else False,\n        "unique_per_dataset_eval": args.unique_per_dataset_eval if dataset_name is not None else False,\n        "sampler_num_iters": batch_sampler.num_iters,\n        "sampler_dataset_lengths": batch_sampler._dataset_lengths,\n        "sampler_weights": batch_sampler.weights,\n        "ablation": {\n            "ablate_state": args.ablate_state,\n            "ablate_fov": args.ablate_fov,\n            "repeated_diffusion_steps": configs.get("repeated_diffusion_steps"),\n        },\n        "metrics": {},\n    }\n    for key, values in component_values.items():\n        array = np.asarray(values, dtype=np.float64)\n        summary["metrics"][key] = {\n            "mean": float(array.mean()),\n            "std": float(array.std()),\n            "min": float(array.min()),\n            "max": float(array.max()),\n            "p10": float(np.quantile(array, 0.10)),\n            "p50": float(np.quantile(array, 0.50)),\n            "p90": float(np.quantile(array, 0.90)),\n        }\n\n    summary_path = output_jsonl.with_suffix(".summary.json")\n    with open(summary_path, "w") as f:\n        json.dump(summary, f, indent=2)\n\n    del model\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    return summary\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Evaluate VITRA weights.pt loss on a deterministic sampler slice.")\n    parser.add_argument("--config", required=True, type=str, help="Training config JSON/YAML used for the run.")\n    parser.add_argument(\n        "--weights",\n        default=None,\n        type=str,\n        help="Path to weights.pt or a checkpoint directory containing weights.pt. Not required for --dry_run.",\n    )\n    parser.add_argument("--output_jsonl", default=".tmp/eval_loss.jsonl", type=str, help="Path for per-batch JSONL metrics.")\n    parser.add_argument("--eval_dataset", default=None, type=str, help="Evaluate one dataset from the configured mixture, e.g. epic or ssv2.")\n    parser.add_argument("--eval_each_dataset", action="store_true", help="Evaluate each dataset in the configured mixture separately.")\n    parser.add_argument("--eval_epoch", default=0, type=int, help="Sampler epoch to evaluate.")\n    parser.add_argument("--eval_sampler_step", default=20000, type=int, help="Sampler micro-batch step to start evaluation from.")\n    parser.add_argument("--eval_batches", default=200, type=int, help="Number of sampler batches to evaluate.")\n    parser.add_argument("--seen_epoch", default=0, type=int, help="Sampler epoch for the seen slice overlap check.")\n    parser.add_argument("--seen_sampler_steps", default=None, type=int, help="If set, assert eval ids do not overlap sampler steps [0, N).")\n    parser.add_argument("--seen_optimizer_steps", default=None, type=int, help="Convert optimizer steps to seen sampler steps using gradient accumulation.")\n    parser.add_argument("--grad_accumulation_steps", default=None, type=int, help="Override gradient accumulation when using --seen_optimizer_steps.")\n    parser.add_argument(\n        "--allow_seen_overlap",\n        action="store_true",\n        help="Allow raw sample-id overlap in mixed weighted evaluation. Useful because weighted oversampling can repeat ids.",\n    )\n    parser.add_argument(\n        "--exclude_seen_ids",\n        action=argparse.BooleanOptionalAction,\n        default=True,\n        help="For per-dataset evaluation, skip raw sample ids consumed in the seen sampler slice.",\n    )\n    parser.add_argument(\n        "--unique_per_dataset_eval",\n        action=argparse.BooleanOptionalAction,\n        default=True,\n        help="For per-dataset evaluation, avoid duplicate raw sample ids in the eval batches.",\n    )\n    parser.add_argument("--data_mix", default=None, type=str, help="Override train_dataset.data_mix.")\n    parser.add_argument("--seed", default=None, type=int, help="Override config seed.")\n    parser.add_argument("--batch_size", default=None, type=int, help="Override per-device batch size.")\n    parser.add_argument("--total_batch_size", default=None, type=int, help="Override global batch size for metadata consistency.")\n    parser.add_argument("--fwd_pred_next_n", default=None, type=int, help="Override future action horizon.")\n    parser.add_argument("--repeated_diffusion_steps", default=None, type=int, help="Override repeated diffusion steps for forward-loss evaluation.")\n    parser.add_argument("--num_workers", default=None, type=int, help="Override DataLoader workers.")\n    parser.add_argument("--prefetch_factor", default=None, type=int, help="Override DataLoader prefetch factor.")\n    parser.add_argument("--use_bf16", action="store_true", help="Force model use_bf16=True.")\n    parser.add_argument(\n        "--ablate_state",\n        default="none",\n        choices=["none", "zero_state", "no_state", "shuffle_state"],\n        help=(\n            "Inference-only state ablation. zero_state zeros state values but keeps masks; "\n            "no_state zeros values and masks; shuffle_state cyclically shifts states within each batch."\n        ),\n    )\n    parser.add_argument(\n        "--ablate_fov",\n        default="none",\n        choices=["none", "zero_fov", "shuffle_fov"],\n        help="Inference-only FOV ablation. shuffle_fov cyclically shifts FOV values within each batch.",\n    )\n    parser.add_argument(\n        "--keep_train_augmentation",\n        action="store_true",\n        help="Keep train_dataset augmentation/state masking settings. Default disables stochastic eval transforms.",\n    )\n    # ── Dry-run / counting mode ────────────────────────────────────────────────\n    parser.add_argument(\n        "--dry_run",\n        action="store_true",\n        help=(\n            "Count unique unseen samples per dataset for each cutoff in --sweep_cutoff_steps "\n            "without loading the model. For each cutoff C, samples from [0, C) are treated as "\n            "seen (inferred automatically); samples from [C, epoch-end) that are unique and "\n            "not in the seen set are counted. Total corpus size is derived from the dataset "\n            "lengths. --seen_sampler_steps is ignored in this mode."\n        ),\n    )\n    parser.add_argument(\n        "--sweep_cutoff_steps",\n        default=None,\n        type=str,\n        help=(\n            "Comma-separated list of sampler cutoff steps to sweep in --dry_run mode "\n            "(overrides --eval_sampler_step for each entry). "\n            "Example: --sweep_cutoff_steps 64000,80000,96000,112000"\n        ),\n    )\n    return parser.parse_args()\n\n\ndef run_dry_run(configs: Dict[str, Any], args: argparse.Namespace) -> None:\n    """Count unique unseen samples available per dataset from one or more cutoff steps.\n\n    Builds the sampler once (no model load). For each cutoff step C:\n      - seen_ids are collected from sampler steps [0, C) automatically.\n      - count_available_samples walks [C, epoch-end) and counts unique unseen samples.\n      - total_corpus_size is derived from sum(batch_sampler._dataset_lengths).\n\n    Prints a table of (cutoff, total_available, fraction_of_corpus, per_dataset)\n    so you can identify which 16k-step checkpoint boundary leaves ~20% for eval.\n    """\n    import time\n\n    overwatch.info("[dry_run] Building dataset and sampler (no model load) …")\n    t0 = time.monotonic()\n\n    # Build only the sampler; we need a processor to call get_vla_dataset_and_collator.\n    # Construct the model CPU-only just to extract the processor, then discard it.\n    from vitra.models.vla_builder import build_vla  # noqa: F811\n\n    tmp_model = build_vla(configs=copy.deepcopy(configs))\n    processor = tmp_model.processor\n    del tmp_model\n\n    vla_dataset, _collator, batch_sampler = make_dataset_and_sampler(configs, processor, args)\n    dataset_names = get_dataset_names(vla_dataset)\n    num_datasets = len(dataset_names)\n\n    # Total unique corpus size is the sum of per-dataset lengths from the sampler.\n    total_corpus = sum(batch_sampler._dataset_lengths)\n    overwatch.info(\n        f"[dry_run] Sampler ready in {time.monotonic() - t0:.1f}s — "\n        f"{num_datasets} datasets: {dataset_names}, "\n        f"total_corpus={total_corpus:,}"\n    )\n\n    # Determine which cutoff steps to sweep.\n    if args.sweep_cutoff_steps is not None:\n        cutoff_steps = [int(s.strip()) for s in args.sweep_cutoff_steps.split(",")]\n    else:\n        cutoff_steps = [args.eval_sampler_step]\n\n    results = []\n    for cutoff in cutoff_steps:\n        # Seen samples = everything the model was trained on up to this checkpoint.\n        # Inferred directly from the cutoff step; --seen_sampler_steps is ignored here.\n        t1 = time.monotonic()\n        overwatch.info(f"[dry_run] cutoff={cutoff}: collecting seen_ids from [0, {cutoff}) …")\n        seen_ids = collect_sampler_ids(batch_sampler, args.seen_epoch, 0, cutoff)\n        overwatch.info(\n            f"[dry_run] cutoff={cutoff}: {len(seen_ids):,} seen sample-ids "\n            f"in {time.monotonic() - t1:.1f}s"\n        )\n\n        t2 = time.monotonic()\n        counts = count_available_samples(\n            batch_sampler,\n            epoch=args.eval_epoch,\n            cutoff_step=cutoff,\n            num_datasets=num_datasets,\n            seen_ids=seen_ids if args.exclude_seen_ids else set(),\n            exclude_seen=args.exclude_seen_ids,\n        )\n        elapsed = time.monotonic() - t2\n        total_available = sum(counts.values())\n        fraction = round(total_available / total_corpus, 4) if total_corpus > 0 else None\n        entry = {\n            "cutoff_step": cutoff,\n            "seen_ids_count": len(seen_ids),\n            "total_available": total_available,\n            "fraction_of_corpus": fraction,\n            "elapsed_s": round(elapsed, 2),\n            "per_dataset": {dataset_names[i]: counts[i] for i in range(num_datasets)},\n        }\n        results.append(entry)\n        overwatch.info(\n            f"[dry_run] cutoff={cutoff}: available={total_available:,} "\n            f"({fraction:.1%} of corpus) in {elapsed:.1f}s"\n        )\n        for name, cnt in entry["per_dataset"].items():\n            overwatch.info(f"  {name}: {cnt:,}")\n\n    # Write JSON summary.\n    output_path = Path(args.output_jsonl).with_suffix(".dry_run.json")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    dry_run_summary = {\n        "mode": "dry_run",\n        "exclude_seen_ids": args.exclude_seen_ids,\n        "dataset_names": dataset_names,\n        "total_corpus": total_corpus,\n        "sampler_num_iters": batch_sampler.num_iters,\n        "sampler_dataset_lengths": batch_sampler._dataset_lengths,\n        "sampler_weights": batch_sampler.weights,\n        "cutoff_sweep": results,\n    }\n    with open(output_path, "w") as f:\n        json.dump(dry_run_summary, f, indent=2)\n    overwatch.info(f"[dry_run] Summary written to {output_path}")\n    if overwatch.is_rank_zero():\n        print(json.dumps(dry_run_summary, indent=2))\n\n\nif __name__ == "__main__":\n    if not dist.is_initialized():\n        os.environ.setdefault("RANK", "0")\n        os.environ.setdefault("WORLD_SIZE", "1")\n        os.environ.setdefault("LOCAL_RANK", "0")\n        os.environ.setdefault("MASTER_ADDR", "localhost")\n        os.environ.setdefault("MASTER_PORT", "29501")\n        backend = "nccl" if torch.cuda.is_available() else "gloo"\n        dist.init_process_group(backend=backend)\n\n    args = parse_args()\n\n    # In dry_run mode --weights is not required.\n    if not args.dry_run and args.weights is None:\n        raise ValueError("--weights is required unless --dry_run is set.")\n\n    # Patch resolve_weights_path to be a no-op when weights not provided.\n    if args.weights is None:\n        args.weights = "__dry_run_placeholder__"\n\n    configs = update_configs(load_config(args.config), args)\n\n    if args.seen_optimizer_steps is not None and args.seen_sampler_steps is None:\n        grad_accumulation_steps = args.grad_accumulation_steps\n        if grad_accumulation_steps is None:\n            grad_accumulation_steps = configs["total_batch_size"] // configs["batch_size"] // overwatch.world_size()\n        args.seen_sampler_steps = args.seen_optimizer_steps * grad_accumulation_steps\n\n    if args.eval_each_dataset and args.eval_dataset is not None:\n        raise ValueError("Use either --eval_each_dataset or --eval_dataset, not both.")\n\n    if args.dry_run:\n        run_dry_run(configs, args)\n    elif args.eval_each_dataset:\n        dataset_names = get_config_dataset_names(configs)\n        summaries = [evaluate(configs, args, dataset_name=name) for name in dataset_names]\n        summary = {\n            "datasets": [item["dataset"] for item in summaries],\n            "summary_paths": [\n                str(Path(args.output_jsonl).with_name(f"{Path(args.output_jsonl).stem}.{item[\'dataset\']}{Path(args.output_jsonl).suffix}").with_suffix(".summary.json"))\n                for item in summaries\n            ],\n            "metrics": {item["dataset"]: item["metrics"] for item in summaries},\n        }\n        if overwatch.is_rank_zero():\n            print(json.dumps({"summary_paths": summary["summary_paths"], "metrics": summary["metrics"]}, indent=2))\n    else:\n        summary = evaluate(configs, args, dataset_name=args.eval_dataset)\n        if overwatch.is_rank_zero():\n            if args.eval_dataset is not None:\n                output_jsonl = Path(args.output_jsonl)\n                summary_path = str(output_jsonl.with_name(f"{output_jsonl.stem}.{args.eval_dataset}{output_jsonl.suffix}").with_suffix(".summary.json"))\n            else:\n                summary_path = str(Path(args.output_jsonl).with_suffix(".summary.json"))\n            print(json.dumps({"summary_path": summary_path, "metrics": summary["metrics"]}, indent=2))\n\n    if dist.is_initialized():\n        dist.barrier()\n        dist.destroy_process_group()\n'

Path("scripts/evaluate_pretrained_loss.py").write_text(EVAL_SCRIPT, encoding="utf-8")

DATA_MIXTURE = 'from typing import Dict, List, Tuple\n\nHAND_MIXTURES: Dict[str, List[Tuple[str, float]]] = {\n    # === ego4d + egoexo4d + ssv2 + epic Dataset ===\n    "magic_mix": [\n        ("ego4d_cooking_and_cleaning", 1.0),\n        ("egoexo4d", 3.0),\n        ("epic", 1.0),\n        (\'ssv2\', 5.0),\n        (\'ego4d_other\', 0.5)\n    ],\n    "magic_mix_cooking_and_cleaning": [\n        ("ego4d_cooking_and_cleaning", 1.0),\n        ("egoexo4d", 3.0),\n        ("epic", 1.0),\n        (\'ssv2\', 5.0),\n    ],\n    "magic_mix_epic_ssv2": [\n        ("epic", 1.0),\n        (\'ssv2\', 5.0),\n    ],\n}\n'
Path("vitra/datasets/data_mixture.py").write_text(DATA_MIXTURE, encoding="utf-8")

HUMAN_DATASET = 'import bisect\nimport copy\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom functools import lru_cache\nfrom typing import Optional\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom PIL import Image\nfrom scipy.spatial.transform import Rotation as R\nfrom torch.utils.data import Dataset\nfrom tqdm import tqdm\nfrom transformers import PreTrainedTokenizerBase\n\nfrom vitra.datasets.augment_utils import (\n    augmentation_func,\n    center_crop_short_side,\n    project_to_image_space,\n)\n\nfrom vitra.datasets.interp_utils import interp_mano_state\nfrom vitra.datasets.video_utils import load_video_decord\nfrom vitra.datasets.dataset_utils import (\n    compute_new_intrinsics_crop,\n    compute_new_intrinsics_resize,\n    calculate_fov,\n    ActionFeature,\n    StateFeature,\n)\nfrom vitra.utils.data_utils import (\n    read_dataset_statistics,\n    GaussianNormalizer,\n)\n\nclass EpisodicDatasetCore(object):\n    """Core dataset class for episodic hand manipulation data.\n\n    Handles loading and processing of video frames, MANO hand parameters,\n    and action sequences for hand-centric manipulation tasks.\n    """\n    def __init__(\n        self,\n        video_root,\n        annotation_file,\n        label_folder,\n        training_path=None,\n        statistics_path=None,\n        augmentation=True,\n        flip_augmentation=True,\n        set_none_ratio=0.0,\n        action_type="angle",\n        use_rel=False,\n        upsample_factor=1.0,\n        target_image_height=224,\n        clip_len=2000,\n        state_mask_prob=0.1,\n        action_past_window_size=0,\n        action_future_window_size=15,\n        image_past_window_size=0,\n        image_future_window_size=0,\n        rel_mode="step",\n        load_images=True,\n    ):\n        self.video_root = video_root\n        annotation_dict = np.load(annotation_file, allow_pickle=True)\n        self.label_folder = label_folder\n        self.index_frame_pair = annotation_dict[\'index_frame_pair\'].copy()\n        self.index_to_episode_id = annotation_dict[\'index_to_episode_id\'].copy()\n\n        if training_path is not None:\n            self.training_idx = np.load(training_path, allow_pickle=True)\n            self.num_valid_frames = len(self.training_idx)\n        else:\n            self.training_idx = None\n            self.num_valid_frames = len(self.index_frame_pair)\n\n        if statistics_path is not None:\n            self.data_statistics = read_dataset_statistics(statistics_path)\n\n        print("Filtering missing videos...")\n        available_videos_cache = {}\n        valid_indices = []\n\n        for idx in tqdm(range(self.num_valid_frames), desc="Filtering indices"):\n            corr = self.index_frame_pair[idx]\n            episode_id = self.index_to_episode_id[corr[0]]\n            dataset_name = episode_id.split(\'_\')[0]\n\n            if dataset_name == \'epic\':\n                # Load episode npy to get video_name\n                video_name = \'_\'.join(episode_id.split(\'_\')[2:4])\n\n                # Check if we already verified this video\n                if video_name not in available_videos_cache:\n                    # Construct expected path (adjust for part_index if clip_len is used)\n                    expected_path = os.path.join(self.video_root, video_name + \'.MP4\')\n                    available_videos_cache[video_name] = os.path.exists(expected_path)\n\n                if not available_videos_cache[video_name]:\n                    continue  # Skip this frame, video is missing\n\n            valid_indices.append(idx)\n\n        # Reassign the filtered indices\n        if self.training_idx is not None:\n            self.training_idx = np.array(valid_indices)\n        else:\n            self.index_frame_pair = self.index_frame_pair[valid_indices]\n\n        self.num_valid_frames = len(valid_indices)\n        print(f"Filtered to {self.num_valid_frames} frames.")\n\n        self.global_data_statistics = None\n        self.clip_len = clip_len  # Video clip length in frames\n        self.augmentation = augmentation\n        self.target_image_height = target_image_height\n        self.flip_augmentation = flip_augmentation\n        self.set_none_ratio = set_none_ratio\n        self.action_type = action_type  # "angle" (Euler angles) or "keypoints" (3D joint positions)\n        self.use_rel = use_rel  # Whether to use relative delta as actions for hand poses (MANO poses)\n        assert upsample_factor >= 1.0, "only support upsample_factor >= 1.0"\n        self.upsample_factor = upsample_factor\n        self.state_mask_prob = state_mask_prob  # Probability of masking state input\n\n        self.action_past_window_size=action_past_window_size\n        self.action_future_window_size=action_future_window_size\n        self.image_past_window_size=image_past_window_size\n        self.image_future_window_size=image_future_window_size\n        self.rel_mode=rel_mode\n        self.load_images=load_images\n\n    def __len__(self):\n        return self.num_valid_frames\n\n    @staticmethod\n    @lru_cache(maxsize=256)          # ~256 MB worst case if each npy ≈1 MB\n    def _load_episode_npy(episode_path: str):\n        """Load episode data from .npy file with caching.\n\n        Uses LRU cache to keep up to 256 episodes in memory (~256 MB worst case).\n        The cache automatically purges old entries when full.\n\n        Args:\n            episode_path: Path to the .npy file containing episode data\n\n        Returns:\n            Dictionary containing episode information\n        """\n        return np.load(episode_path, allow_pickle=True).item()\n\n    def _load_or_cache_episode(self, episode_id):\n        """\n        Returns episode_result (raw dict) and the pre-extracted camera\n        extrinsics (R_w2c, t_w2c).  No camera-space MANO tensors are cached.\n        """\n        root = self.label_folder\n        epi_path = os.path.join(root, episode_id + \'.npy\')\n        epi = self._load_episode_npy(epi_path)\n\n        extr = epi[\'extrinsics\']                         # world to cam, (T,4,4)\n        R_w2c, t_w2c = extr[:, :3, :3], extr[:, :3, 3]\n\n        return epi, R_w2c, t_w2c\n\n    def _mat2euler(self, R_batch: np.ndarray) -> np.ndarray:\n        """Batched XYZ-Euler conversion using SciPy."""\n        flat = R_batch.reshape(-1, 3, 3)\n        eul = R.from_matrix(flat).as_euler(\'xyz\', degrees=False)\n        return eul\n\n    def _prepare_side_window(self, side_dict,\n                            R_w2c, t_w2c,\n                            idx_window, idx_anchor,\n                            *, anchor_frame=True, oob=None, start=None, end=None, upsample_factor=1.0):\n\n        T, W = len(side_dict[\'global_orient_worldspace\']), len(idx_window)\n        idx_window_extend = np.append(idx_window, np.clip(idx_window[-1] + 1, start, end))\n\n        kept_extend = side_dict[\'kept_frames\'][idx_window_extend].astype(bool)\n\n        R_mano_extend = side_dict[\'global_orient_worldspace\'][idx_window_extend]\n        t_mano_extend = side_dict[\'transl_worldspace\'][idx_window_extend]\n        hand_P_extend = side_dict[\'hand_pose\'][idx_window_extend]\n        joints_worldspace_extend = side_dict[\'joints_worldspace\'][idx_window_extend]\n\n        oob_indices = np.where(oob)[0]\n        if len(oob_indices) > 0:\n            # If more than 0 frames are out of bounds, set kept to False\n            kept_extend[oob_indices] = False\n\n            # consider OOB for the last frame\n            if idx_window[-1] + 1 > end:\n                kept_extend[-1] = False\n\n        if not np.all(kept_extend):\n            identity = np.eye(3, dtype=hand_P_extend.dtype)\n            identity_block = np.broadcast_to(identity, (hand_P_extend.shape[1], 3, 3))\n            hand_P_extend[~kept_extend] = identity_block\n            R_mano_extend[~kept_extend] = identity\n\n        # -------- camera-space (anchor camera) ----------------------------\n        R_cam_extend  = R_w2c[idx_anchor] @ R_mano_extend\n        t_cam_extend  = (R_w2c[idx_anchor] @ t_mano_extend[..., None])[..., 0] + t_w2c[idx_anchor]\n\n\n        # -------- finger Euler (batched) ----------------------------------\n        pose_euler_extend  = R.from_matrix(hand_P_extend.reshape(-1,3,3))     \\\n                            .as_euler(\'xyz\', degrees=False).reshape(-1,45)\n\n        # -------- keypoints in mano space (batched) ----------------------------------\n        joints_manospace_extend = (R_mano_extend.transpose(0, 2, 1) @ (joints_worldspace_extend.transpose(0, 2, 1) - t_mano_extend[..., None])).transpose(0,2,1)  # (W+1,21,3)\n\n        if upsample_factor > 1:\n            R_cam_extend, t_cam_extend, hand_P_extend, joints_manospace_extend, kept_extend = \\\n            interp_mano_state(R_cam_extend, t_cam_extend, hand_P_extend, joints_manospace_extend, kept_extend, upsample_factor, method="pchip")\n\n            pose_euler_extend = R.from_matrix(hand_P_extend.reshape(-1,3,3))     \\\n                            .as_euler(\'xyz\', degrees=False).reshape(-1,45)\n\n            # set length back to W+1\n            R_cam_extend = R_cam_extend[:W+1]\n            t_cam_extend = t_cam_extend[:W+1]\n            hand_P_extend = hand_P_extend[:W+1]\n            pose_euler_extend = pose_euler_extend[:W+1]\n            joints_manospace_extend = joints_manospace_extend[:W+1]\n            kept_extend = kept_extend[:W+1]\n\n        R_cam = R_cam_extend[:-1]\n        t_cam = t_cam_extend[:-1]\n        pose_euler = pose_euler_extend[:-1]\n        hand_P = hand_P_extend[:-1]\n        joints_manospace = joints_manospace_extend[:-1]\n        kept = kept_extend[:-1]\n\n        R_cam_next = R_cam_extend[1:]\n        t_cam_next = t_cam_extend[1:]\n        pose_euler_next = pose_euler_extend[1:]\n        hand_P_next = hand_P_extend[1:]\n        joints_manospace_next = joints_manospace_extend[1:]\n        kept_next = kept_extend[1:]\n\n        return dict(\n            # current frame tensors\n            R_cam=R_cam.astype(np.float32),\n            t_cam=t_cam.astype(np.float32),\n            pose_euler=pose_euler.astype(np.float32),\n            hand_P=hand_P.astype(np.float32),\n            joints_manospace = joints_manospace.astype(np.float32),\n            kept=kept,\n\n            # next-frame tensors (same length W)\n            R_cam_next = R_cam_next.astype(np.float32),\n            t_cam_next = t_cam_next.astype(np.float32),\n            pose_euler_next = pose_euler_next.astype(np.float32),\n            hand_P_next = hand_P_next.astype(np.float32),\n            joints_manospace_next = joints_manospace_next.astype(np.float32),\n            kept_next = kept_next,\n        )\n    # ============================================================\n    #  4.  Vectorised action-window constructor (ONE hand)\n    # ============================================================\n    def _make_action_window_vec(self, win, anchor_idx: int, *, rel_mode="step", action_type="angle"):\n        """\n        anchor_idx : the position of t0 inside the window (usually = past)\n        rel_mode   : "step"   → Δ(t→t+1)\n                    "anchor" → Δ(t→t0)\n        action_type: "angle"  → Euler angles (xyz)\n                    "keypoints " → root keypoints (21x3=63)\n        """\n\n        R_cur, t_cur  = win[\'R_cam\'],        win[\'t_cam\']\n        R_nxt, t_nxt  = win[\'R_cam_next\'],   win[\'t_cam_next\']\n        P_cur, P_nxt  = win[\'hand_P\'],       win[\'hand_P_next\']\n        pose_next     = win[\'pose_euler_next\']\n        kpoints_root_next = win[\'joints_manospace_next\']\n        kept, kept_n  = win[\'kept\'],         win[\'kept_next\']\n        W = len(t_cur)\n\n        # absolute pose of t+1\n        if action_type == "keypoints":\n            abs_next = kpoints_root_next.reshape(W, -1)\n        elif action_type == "angle":\n            abs_next = pose_next\n        action_abs = np.concatenate(\n            [t_nxt,\n            self._mat2euler(R_nxt),\n            abs_next],\n            axis=-1).astype(np.float32)\n        action_abs = action_abs.reshape(W, -1)\n\n        # choose relative formulation\n        if rel_mode == "step":\n            t_rel = t_nxt - t_cur\n            R_rel = R_nxt @ R_cur.transpose(0,2,1)\n            P_rel = np.matmul(P_nxt, P_cur.transpose(0,1,3,2))\n            valid = kept & kept_n\n\n        elif rel_mode == "anchor":\n            t_anchor  = t_cur[anchor_idx]\n            R_anchor  = R_cur[anchor_idx]\n            P_anchor  = P_cur[anchor_idx]\n\n            # broadcast to all W rows\n            t_rel = t_nxt - t_anchor\n            R_rel = R_nxt @ R_anchor.T\n            P_rel = np.matmul(P_nxt, P_anchor.transpose(0,2,1))\n            valid = kept_n & kept[anchor_idx]\n\n        else:\n            raise ValueError(\'rel_mode must be "step" or "anchor"\')\n\n        pose_rel = R.from_matrix(P_rel.reshape(-1,3,3)) \\\n                    .as_euler(\'xyz\',False).reshape(W,45)\n\n        action_rel = np.concatenate(\n            [t_rel,\n            self._mat2euler(R_rel),\n            pose_rel],\n            axis=-1).astype(np.float32)\n\n        action_abs[~valid] = 0.0\n        action_rel[~valid] = 0.0\n        return action_abs, action_rel, valid\n\n    def _window_indices(self, frame_id, past, future, start, end):\n        """\n        Returns:\n            idx_clip : (W,) indices clipped to [0, T-1]\n            oob      : (W,) bool — slots that were originally OOB\n        """\n        win = np.arange(-past, future + 1) + frame_id                  # (W,)\n        oob = (win < start) | (win > end)\n        return win.clip(start, end), oob\n\n    def _resolve_video_path(self, dataset_name: str = None, video_name: str = None, part_index: int = None) -> str:\n        if dataset_name==\'Ego4D\':\n            if self.clip_len is not None:\n                video_path = os.path.join(self.video_root, video_name + \'_part\' + str(part_index+1) +\'.mp4\')\n            else:\n                video_path = os.path.join(self.video_root, video_name +\'.mp4\')\n            return video_path\n        elif dataset_name==\'EgoExo4D\':\n            if self.clip_len is not None:\n                video_path = os.path.join(self.video_root, video_name +\'_part\' + str(part_index+1) +\'.mp4\')\n            else:\n                video_path = os.path.join(self.video_root, video_name +\'.mp4\')\n            return video_path\n        elif dataset_name == \'epic\':\n            video_id = video_name.split(\'_\')[0]\n            if self.clip_len is not None:\n                video_path = os.path.join(self.video_root, video_name+ \'_part\' + str(part_index+1) + \'.mp4\')\n            else:\n                video_path = os.path.join(self.video_root, video_name+ \'.MP4\')\n            return video_path\n        elif dataset_name == \'somethingsomethingv2\':\n            if self.clip_len is not None:\n                video_path = os.path.join(self.video_root, video_name+ \'_part\' + str(part_index+1) + \'.mp4\')\n            else:\n                video_path = os.path.join(self.video_root, video_name+\'.webm\')\n            return video_path\n        else:\n            raise ValueError(f\'Unknown dataset prefix {dataset_name}\')\n\n    def _mano_forward(self, betas, pose_m):\n        """Runs MANO once and returns (vertices, joints) on CPU NumPy."""\n        beta_t  = torch.tensor(betas).unsqueeze(0).float().cuda()\n        pose_t  = torch.tensor(pose_m).unsqueeze(0).float().cuda()\n        out     = mano(betas=beta_t, hand_pose=pose_t)       # no global_orient\n        return out.vertices[0].cpu().numpy(), out.joints[0].cpu().numpy()\n\n    # ------------------------------------------------------------\n    #  Grab the (past + future + 1) frame window\n    # ------------------------------------------------------------\n\n    def _pack_state(self, R_cam, t_cam, pose_euler, idx):\n        return np.concatenate([t_cam[idx],\n                            self._mat2euler(R_cam[idx][None,...])[0],\n                            pose_euler[idx]])\n\n    def _grab_window_images(self,\n                            episode_id: str,\n                            epi: dict,\n                            frame_id: int,\n                            past: int,\n                            future: int\n                            ):\n        """\n        Returns\n        -------\n        images : (L, H, W, 3)  uint8   – raw RGB frames\n        mask   : (L,) bool                True where real frame, False where pad\n        """\n        dataset_name = episode_id.split(\'_\')[0]\n        # video_path   = self._resolve_video_path(dataset_name, epi[\'video_name\'])\n        decode_table = epi[\'video_decode_frame\']                      # (T,)\n        T            = len(decode_table)\n        frame_in_video = epi[\'video_decode_frame\'][frame_id]\n\n        # ---------- build padded window indices ---------------------------\n        # Not support multiple images now\n        idx_win, oob = self._window_indices(frame_id, past, future, 0, T-1)     # (W,)\n\n        if self.clip_len is not None:\n            part_idx = frame_in_video // self.clip_len # clip_len = 2000\n            frame_in_part = frame_in_video % self.clip_len\n            video_path = self._resolve_video_path(dataset_name, epi[\'video_name\'], part_idx)\n            decode_ids = [frame_in_part]\n        else:\n            video_path = self._resolve_video_path(dataset_name, epi[\'video_name\'])\n            decode_ids = decode_table[idx_win]\n\n        # ---------- read images --------------------\n        # Retry mechanism: try up to 3 times to load video frames\n        imgs = None\n        for attempt in range(3):\n            try:\n                imgs, _ = load_video_decord(video_path, frame_index=decode_ids, rotation=False)\n                break  # Success, exit the retry loop\n            except Exception as e:\n                print(f"Warning: failed to load video frames from {video_path} (attempt {attempt+1}/3): {e}")\n                time.sleep(0.1)\n\n        if imgs is None:\n            raise RuntimeError(f"Failed to load video frames from {video_path} after 3 attempts")\n\n        images = np.stack(imgs, axis=0)           # (L,H,W,3) uint8\n        mask   = ~oob                             # (L,) bool\n\n        return images, mask\n\n    def _find_matching_texts(self, text_list, frame_id):\n        """Find text annotations that overlap with the given frame.\n\n        Args:\n            text_list: List of tuples (text, (start_frame, end_frame))\n            frame_id: Current frame ID to check\n\n        Returns:\n            matching_texts: List of matching text annotations\n            matching_ranges: List of corresponding time ranges (start_frame, end_frame)\n\n        Note:\n            Uses half-open interval [start_frame, end_frame)\n        """\n        matching_texts = []\n        matching_ranges = []\n\n        for text, (start_frame, end_frame) in text_list:\n            # Check if frame_id is in the half-open interval [start_frame, end_frame)\n            if start_frame <= frame_id < end_frame:\n                matching_texts.append(text)\n                matching_ranges.append((start_frame, end_frame))\n\n        return matching_texts, matching_ranges\n\n    def _random_select_text(\n        self,\n        text,\n        text_rephrase,\n        hand_type,\n        clip_idx,\n    ):\n\n        text_list = [text]\n        if text_rephrase and isinstance(text_rephrase[hand_type][clip_idx][0], list):\n            text_list.extend(text_rephrase[hand_type][clip_idx][0])\n\n        text_selected = random.choice(text_list).strip()\n        if not text_selected.endswith(\'.\'):\n            text_selected += \'.\'\n        return text_selected\n\n    def _build_instruction(\n        self,\n        main_type,\n        text_clip,\n        text_rephrase,\n        idx_win,\n        oob,\n        epi_len, # T\n        frame_id,\n        action_past_window_size,\n        action_future_window_size,\n    ):\n\n        sub_type = \'right\' if main_type == \'left\' else \'left\'\n\n        # Build main text\n        # text_clip[main_type][0]: (\'Place the pink cup on the table.\', (0, 26))\n        main_text_selected = self._random_select_text(\n            text_clip[main_type][0][0],\n            text_rephrase,\n            main_type,\n            clip_idx=0,\n        )\n\n        # Build sub text\n        sub_text_list = text_clip[sub_type]\n        has_sub_text = len(sub_text_list) > 0\n\n        sub_oob, sub_idx_win = oob, idx_win\n        sub_text_selected = "None."\n        sub_win = (0, epi_len)  # Default to the full range if no text available\n\n        if has_sub_text:\n            sub_matching_texts, sub_matching_ranges = self._find_matching_texts(sub_text_list, frame_id)\n            if len(sub_matching_texts) > 0:\n                selected_idx = random.randrange(len(sub_matching_texts))\n                sub_win = sub_matching_ranges[selected_idx]\n                sub_idx_win, sub_oob = self._window_indices(\n                    frame_id,\n                    action_past_window_size,\n                    action_future_window_size, sub_win[0], sub_win[1]-1\n                )     # (W,)\n\n                sub_text_selected = self._random_select_text(\n                    sub_matching_texts[selected_idx].strip(),\n                    text_rephrase,\n                    sub_type,\n                    clip_idx=selected_idx,\n                )\n\n        # Assign left/right based on main_type\n        is_main_left = (main_type == \'left\')\n\n        idx_win_left = idx_win if is_main_left else sub_idx_win\n        idx_win_right = sub_idx_win if is_main_left else idx_win\n        oob_left = oob if is_main_left else sub_oob\n        oob_right = sub_oob if is_main_left else oob\n\n        text_left = main_text_selected if is_main_left else sub_text_selected\n        text_right = sub_text_selected if is_main_left else main_text_selected\n\n        start_left = 0 if is_main_left else (sub_win[0] if has_sub_text else 0)\n        start_right = (sub_win[0] if has_sub_text else 0) if is_main_left else 0\n        end_left = epi_len - 1 if is_main_left else (sub_win[1] - 1 if has_sub_text else epi_len - 1)\n        end_right = (sub_win[1] - 1 if has_sub_text else epi_len - 1) if is_main_left else epi_len - 1\n\n        instruction = f"Left hand: {text_left} Right hand: {text_right}"\n\n        return instruction, idx_win_left, oob_left, idx_win_right, oob_right, start_left, end_left, start_right, end_right\n\n    def _get_2d_traj_cur_to_end(self, idx_frame, epi, intrinsics, hand_type, image_size):\n        """Get the 2D trajectory of the hand palm from current frame to episode end.\n\n        Args:\n            idx_frame: Current frame index\n            epi: Episode data dictionary\n            intrinsics: Camera intrinsic matrix\n            hand_type: \'left\' or \'right\' hand\n            image_size: (H, W) tuple of image dimensions\n\n        Returns:\n            Normalized 2D palm trajectory in image space [0, 1]\n        """\n        H, W = image_size\n        # intrinsics = epi[\'intrinsics\'].copy()\n        intrinsics = intrinsics.copy()\n        # normalize intrinsics\n        intrinsics[0] /= intrinsics[0,2]*2\n        intrinsics[1] /= intrinsics[1,2]*2\n\n        hand_joints_cur_to_end = epi[hand_type][\'joints_worldspace\'][idx_frame:] # (N, 21, 3)\n        hand_palm_cur_to_end = np.mean(hand_joints_cur_to_end[:, [0,2,5,9,13,17], :], axis=1, keepdims=True) # (N, 1, 3)\n\n        extrinsics = epi[\'extrinsics\'].copy()\n        extrinsics_cur = extrinsics[idx_frame] # world to cam\n        R_world_to_cam = extrinsics_cur[None, :3, :3].repeat(len(hand_palm_cur_to_end), axis=0)\n        t_world_to_cam = extrinsics_cur[None, :3, 3:].repeat(len(hand_palm_cur_to_end), axis=0)\n\n        hand_palm_cur_to_end_cam = (R_world_to_cam @ hand_palm_cur_to_end.transpose(0, 2, 1) + t_world_to_cam).transpose(0, 2, 1)\n\n        uv_palm_cur_to_end = project_to_image_space(hand_palm_cur_to_end_cam, intrinsics, (H, W)) # (N, M, 2)\n        uv_palm_cur_to_end[..., 0] = np.clip(uv_palm_cur_to_end[..., 0], 0, W)\n        uv_palm_cur_to_end[..., 1] = np.clip(uv_palm_cur_to_end[..., 1], 0, H)\n\n        uv_palm_cur_to_end = uv_palm_cur_to_end.reshape(-1, 2)\n        uv_palm_cur_to_end = uv_palm_cur_to_end.astype(np.float32)\n        uv_palm_cur_to_end[:,0] /= W\n        uv_palm_cur_to_end[:,1] /= H\n\n        return uv_palm_cur_to_end\n\n    def get_item_frame(\n            self, episode_id, frame_id,\n            action_past_window_size=0,\n            action_future_window_size=0,\n            image_past_window_size=0,\n            image_future_window_size=0,\n            rel_mode: str = "step",\n            load_images: bool = True,\n        ):\n        """\n        Vectorised dataloader.\n\n        """\n        # ------------------------------------------------------------------\n        # 1. Load episode dict  +  extrinsics\n        # ------------------------------------------------------------------\n        epi, R_w2c, t_w2c = self._load_or_cache_episode(episode_id)\n        T  = len(epi[\'extrinsics\']) #\n\n        # ------------------------------------------------------------------\n        # 2. Build frame-window indices\n        # ------------------------------------------------------------------\n        idx_win, oob  = self._window_indices(frame_id,\n                                        action_past_window_size,\n                                        action_future_window_size, 0, T-1)     # (W,)\n        W   = len(idx_win)\n        main_type = epi[\'anno_type\']\n        sub_type = \'right\' if main_type == \'left\' else \'left\'\n        # ------------------------------------------------------------------\n        # 3. Build instruction text\n        # ------------------------------------------------------------------\n        instruction, idx_win_left, oob_left, idx_win_right, oob_right, \\\n        start_left, end_left, start_right, end_right = self._build_instruction(\n            main_type = main_type,\n            text_clip = epi[\'text\'],\n            text_rephrase = epi.get(\'text_rephrase\'),\n            idx_win = idx_win,\n            oob = oob,\n            epi_len = T,\n            frame_id = frame_id,\n            action_past_window_size = action_past_window_size,\n            action_future_window_size = action_future_window_size,\n        )\n\n\n        # ------------------------------------------------------------------\n        # 4. Vectorised actions  (left + right)\n        # ------------------------------------------------------------------\n        win_left  = self._prepare_side_window(\n            epi[\'left\'],  R_w2c, t_w2c, idx_win_left, frame_id, anchor_frame=True,\n            oob=oob_left, start=start_left, end=end_left, upsample_factor=self.upsample_factor\n        )\n        win_right = self._prepare_side_window(\n            epi[\'right\'], R_w2c, t_w2c, idx_win_right, frame_id, anchor_frame=True,\n            oob=oob_right, start=start_right, end=end_right, upsample_factor=self.upsample_factor\n        )\n        idx_center = action_past_window_size          # local index of t0 in window\n\n        # rel_mode: "step"  or  "anchor" / action_type: "angle" or "keypoints"\n        # step: relative to previous frame, anchor: relative to t0\n        abs_L, rel_L, msk_L = self._make_action_window_vec(\n            win_left,  anchor_idx=idx_center, rel_mode=rel_mode, action_type=self.action_type\n        )\n\n        abs_R, rel_R, msk_R = self._make_action_window_vec(\n            win_right, anchor_idx=idx_center, rel_mode=rel_mode, action_type=self.action_type\n        )\n\n        action_abs = np.concatenate([abs_L, abs_R], axis=1)   # (W,action_dim)\n        action_rel = np.concatenate([rel_L, rel_R], axis=1)   # (W,102)\n        action_mask = np.stack([msk_L, msk_R], axis=1)        # (W,2)\n\n        cur_L = self._pack_state(win_left[\'R_cam\'],\n                    win_left[\'t_cam\'],\n                    win_left[\'pose_euler\'] if self.action_type==\'angle\' else win_left[\'joints_manospace\'].reshape(W, -1),\n                    idx_center)\n\n        cur_R = self._pack_state(win_right[\'R_cam\'],\n                            win_right[\'t_cam\'],\n                            win_right[\'pose_euler\'] if self.action_type==\'angle\' else win_right[\'joints_manospace\'].reshape(W, -1),\n                            idx_center)\n\n        betas_L = epi[\'left\'][\'beta\']\n        betas_R = epi[\'right\'][\'beta\']\n\n        current_state       = np.concatenate([cur_L, betas_L, cur_R, betas_R])          # 2 * (6+action_dim+10,)\n        # current_state_mask  = np.array([msk_L[idx_center],\n        #                                 msk_R[idx_center]])\n        current_state_mask  = np.array([win_left[\'kept\'][idx_center], win_right[\'kept\'][idx_center]])\n\n        # ------------------------------------------------------------------\n        # 5. RGB window\n        # ------------------------------------------------------------------\n        if load_images:\n            image_list, image_mask = self._grab_window_images(\n                episode_id, epi,\n                frame_id,\n                image_past_window_size,\n                image_future_window_size\n            )\n            H = image_list[0].shape[0]\n            W = image_list[0].shape[1]\n        else:\n            image_list = None\n            image_mask = None\n            H, W = epi[\'intrinsics\'][1,2]*2, epi[\'intrinsics\'][0,2]*2\n        # ------------------------------------------------------------------\n        # 6. Calculate New_intrinsics\n        # ------------------------------------------------------------------\n        dataset_name = episode_id.split(\'_\')[0]\n        intrinsics = epi[\'intrinsics\']\n\n        if dataset_name == \'EgoExo4D\':\n            # For EgoExo4D, the fisheye camera images contain black borders after undistortion.\n            # We remove these borders using a center crop. Specifically, the video frames are\n            # first resized from 1408 to 448, and then center-cropped to 256.\n\n            new_intrinsics = compute_new_intrinsics_crop(intrinsics, 1408, 256/448*1408, H)\n\n        else:\n            new_intrinsics = compute_new_intrinsics_resize(intrinsics, (H, W))\n\n        # ------------------------------------------------------------------\n        # 7. Do augmentation\n        # ------------------------------------------------------------------\n        if self.augmentation:\n            try:\n                # randomly sample aspect ratio for augmentation\n                aspect_ratio = np.exp(random.uniform(np.log(1.0), np.log(2.0)))\n                target_size = (int(self.target_image_height * aspect_ratio), self.target_image_height)  # (W, H)\n                augment_params = {\n                    \'tgt_aspect\': aspect_ratio,\n                    \'flip_augmentation\': self.flip_augmentation,\n                    \'set_none_ratio\': self.set_none_ratio,\n                }\n\n                uv_traj = self._get_2d_traj_cur_to_end(frame_id, epi, new_intrinsics, main_type, (H, W))\n                image_list, new_intrinsics, (action_abs, action_rel, action_mask), \\\n                (current_state, current_state_mask), instruction = \\\n                    augmentation_func(\n                        image = image_list,\n                        intrinsics = new_intrinsics,\n                        actions = (action_abs, action_rel, action_mask),\n                        states = (current_state, current_state_mask),\n                        captions = instruction,\n                        uv_traj = uv_traj,\n                        target_size = target_size,\n                        augment_params = augment_params,\n                        sub_type = sub_type,\n                    )\n\n            except Exception as e:\n                print(f"Warning: Augmentation failed for episode {episode_id}, frame {frame_id}: {e}. Do center crop only")\n                import traceback\n                print(f"Warning: Augmentation failed for episode {episode_id}, frame {frame_id}")\n                print(f"Exception: {type(e).__name__}: {e}")\n                print(f"Traceback:\\n{traceback.format_exc()}")\n                image_list = center_crop_short_side(image_list[0])[None, ...]\n                new_intrinsics[0][2] = 0.5 * image_list[0].shape[1]  # update the principal point\n                new_intrinsics[1][2] = 0.5 * image_list[0].shape[0]  # update the principal point\n\n            if random.random() < self.state_mask_prob:\n                current_state_mask = np.array([False, False])\n                current_state[:] = 0.0\n\n        fov = calculate_fov( 2 * new_intrinsics[1][2], 2 * new_intrinsics[0][2], new_intrinsics)\n\n        if self.use_rel:\n            action_list = action_rel\n        else:\n            # use abs action for hand pose only\n            rel_L = action_rel[:, :action_rel.shape[1]//2]\n            rel_R = action_rel[:, action_rel.shape[1]//2:]\n            abs_L = action_abs[:, :action_abs.shape[1]//2]\n            abs_R = action_abs[:, action_abs.shape[1]//2:]\n\n            action_list = np.concatenate([rel_L[:, :6], abs_L[:, 6:], rel_R[:, :6], abs_R[:, 6:]], axis=1)\n\n        # ------------------------------------------------------------------\n        # 8. Return to caller\n        # ------------------------------------------------------------------\n\n        result_dict = dict(\n            instruction             = instruction,\n            action_list             = action_list,          # (W,2*51) float32\n            action_mask             = action_mask,          # (W,2)   bool\n            current_state           = current_state,        # (2*61,)  float32\n            current_state_mask      = current_state_mask,   # (2,) bool\n            fov                     = fov,                  # (2,) float32\n            intrinsics              = new_intrinsics,       # (3,3) float32\n        )\n\n        if image_list is not None:\n            result_dict[\'image_list\'] = image_list          # (W,H,W,3) uint8\n        if image_mask is not None:\n            result_dict[\'image_mask\'] = image_mask          # (W,) bool\n\n        return result_dict\n\n    def set_global_data_statistics(self, global_data_statistics):\n        self.global_data_statistics = global_data_statistics\n        if not hasattr(self, \'gaussian_normalizer\'):\n            self.gaussian_normalizer = GaussianNormalizer(self.global_data_statistics)\n\n    def transform_trajectory(\n        self,\n        sample_dict: dict = None,\n        normalization: bool = True,\n    ):\n        """Pad action and state dimensions to match a unified size."""\n        action_np = sample_dict["action_list"]\n        state_np = sample_dict["current_state"]\n\n        action_dim = action_np.shape[1]\n        state_dim = state_np.shape[0]\n        if normalization:\n            # Normalize left and right hand actions and states separately\n            action_np = self.gaussian_normalizer.normalize_action(action_np)\n            state_np = self.gaussian_normalizer.normalize_state(state_np)\n\n        # ===== Pad to unified dimensions =====\n        unified_action_dim = ActionFeature.ALL_FEATURES[1]   # 192\n        unified_state_dim = StateFeature.ALL_FEATURES[1]     # 212\n        unified_state, unified_state_mask = pad_state_human(\n            state_np,\n            sample_dict["current_state_mask"],\n            action_dim,\n            state_dim,\n            unified_state_dim\n        )\n        unified_action, unified_action_mask = pad_action(\n            action_np,\n            sample_dict["action_mask"],\n            action_dim,\n            unified_action_dim\n        )\n\n        sample_dict["action_list"] = unified_action\n        sample_dict["action_mask"] = unified_action_mask\n        sample_dict["current_state"] = unified_state\n        sample_dict["current_state_mask"] = unified_state_mask\n        return sample_dict\n\n    def __getitem__(self, idx):\n        if self.training_idx is not None:\n            data_id = self.training_idx[idx]\n        else:\n            data_id = idx\n        corr = self.index_frame_pair[data_id]\n        episode_id = self.index_to_episode_id[corr[0]]\n        sample = self.get_item_frame(\n            episode_id, int(corr[1]),\n            action_past_window_size=self.action_past_window_size,\n            action_future_window_size=self.action_future_window_size,\n            image_past_window_size=self.image_past_window_size,\n            image_future_window_size=self.image_future_window_size,\n            rel_mode=self.rel_mode,  # \'step\'\n            load_images=self.load_images\n        )\n        return sample\n\ndef pad_state_human(\n    state: torch.Tensor,\n    state_mask: torch.Tensor,\n    action_dim: int,\n    state_dim: int,\n    unified_state_dim: int\n):\n    """\n    Expand state mask, mask invalid state dims, and pad current_state to a standard size.\n\n    Args:\n        current_state (Tensor): original state tensor, shape [state_dim]\n        current_state_mask (Tensor): per-hand state mask, shape [state_dim//2] or [state_dim]\n        action_dim (int): original action dimension\n        state_dim (int): original state dimension\n        unified_state_dim (int): target padded state dimension\n\n    Returns:\n        Tuple[Tensor, Tensor]:\n            padded current_state [unified_state_dim],\n            padded current_state_mask [unified_state_dim]\n    """\n\n    current_state = torch.tensor(state, dtype=torch.float32)\n    current_state_mask = torch.tensor(state_mask, dtype=torch.bool)\n\n    # Expand state mask from per-hand to per-dim\n    expanded_state_mask = current_state_mask.repeat_interleave(state_dim // 2)\n\n    # Mask out invalid state dimensions\n    current_state_masked = current_state * expanded_state_mask.to(current_state.dtype)\n\n    # Initialize output tensors\n    padded_state = torch.zeros(unified_state_dim, dtype=current_state.dtype)\n    padded_mask = torch.zeros(unified_state_dim, dtype=torch.bool)\n\n    # Fill first half of state_dim (left hand), skipping MANO betas\n    padded_state[:action_dim//2] = current_state_masked[:action_dim//2].clone()\n    padded_mask[:action_dim//2] = expanded_state_mask[:action_dim//2].clone()\n\n    # Fill second half of state_dim (right hand), skipping MANO betas\n    padded_state[action_dim//2:action_dim] = current_state_masked[state_dim//2:state_dim//2+action_dim//2].clone()\n    padded_mask[action_dim//2:action_dim] = expanded_state_mask[state_dim//2:state_dim//2+action_dim//2].clone()\n\n    return padded_state, padded_mask\n\ndef pad_action(\n    actions: torch.Tensor = None,\n    action_mask: torch.Tensor = None,\n    action_dim: int = None,\n    unified_action_dim: int = None\n):\n    """\n    Expand action mask per dimension, mask invalid actions, and pad actions to a unified size.\n\n    Args:\n        actions (Tensor or None): original actions tensor, shape [T, action_dim] or None.\n        action_mask (Tensor): per-hand action mask, shape [T, 2].\n        action_dim (int): original action dimension.\n        unified_action_dim (int): target padded actions dimension.\n\n    Returns:\n        Tuple[Optional[Tensor], Tensor]:\n            padded actions [T, unified_action_dim] or None,\n            padded action mask [T, unified_action_dim]\n    """\n\n    action_mask = torch.tensor(action_mask, dtype=torch.bool)\n\n    # Expand mask from per-hand to per-dimension\n    mask_left = action_mask[:, 0].unsqueeze(1).expand(-1, action_dim // 2)\n    mask_right = action_mask[:, 1].unsqueeze(1).expand(-1, action_dim // 2)\n    expanded_action_mask = torch.cat((mask_left, mask_right), dim=1)\n\n    # ---------------------------\n    # Case 1: actions is None\n    # ---------------------------\n    if actions is None:\n        padding_mask = torch.zeros(\n            (action_mask.shape[0], unified_action_dim - action_dim),\n            dtype=torch.bool\n        )\n        action_mask_padded = torch.cat((expanded_action_mask, padding_mask), dim=1)\n        return None, action_mask_padded\n\n    # ---------------------------\n    # Case 2: actions exists\n    # ---------------------------\n\n    actions = torch.tensor(actions, dtype=torch.float32)\n    # Mask invalid action dims\n    actions_masked = actions * expanded_action_mask.to(actions.dtype)\n\n    # Pad both actions and mask\n    padding = torch.zeros(\n        (actions.shape[0], unified_action_dim - action_dim),\n        dtype=actions.dtype\n    )\n\n    actions_padded = torch.cat((actions_masked, padding), dim=1)\n    action_mask_padded = torch.cat((expanded_action_mask, padding.bool()), dim=1)\n\n    return actions_padded, action_mask_padded\n'
Path("vitra/datasets/human_dataset.py").write_text(HUMAN_DATASET, encoding="utf-8")
print("Patched human_dataset.py with missing-video filtering:", "Filtering missing videos" in HUMAN_DATASET)
print("Patched data_mixture.py with magic_mix_epic_ssv2:", "magic_mix_epic_ssv2" in DATA_MIXTURE)
print("Patched scripts/evaluate_pretrained_loss.py with inference-only ablation flags")
print("Contains --ablate_state:", "--ablate_state" in EVAL_SCRIPT)
print("Contains get_local_rank fallback:", "def get_local_rank" in EVAL_SCRIPT)


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from glob import glob
from tqdm import tqdm

PREPROCESS_EPIC = True
EPIC_INPUT_PATTERN = "/kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4"
EPIC_RESIZED_ROOT = Path("/tmp/epic-kitchens-100-224")
EPIC_SHORT_SIDE = 224
EPIC_RESIZE_THREADS = 48


def resize_video_gpu(input_path: str, output_path: str, short_side: int = 224) -> bool:
    """Resize a video so the shorter side equals short_side, preserving aspect ratio."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    scale_filter = f"scale_cuda='if(gt(iw,ih),-2,{short_side})':'if(gt(iw,ih),{short_side},-2)'"
    cmd = [
        "ffmpeg", "-y",
        "-hwaccel", "cuda",
        "-hwaccel_output_format", "cuda",
        "-i", input_path,
        "-vf", scale_filter,
        "-c:v", "h264_nvenc",
        "-cq", "18",
        "-an",
        output_path,
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        print(f"[WARN] Resize failed for {input_path}")
        print(result.stderr.decode("utf-8", errors="ignore")[-1000:])
    return result.returncode == 0


def resize_dataset(input_pattern: str, output_root: Path, short_side: int, num_threads: int):
    videos = sorted(glob(input_pattern, recursive=True))
    output_root.mkdir(parents=True, exist_ok=True)
    tasks = []
    total_size_gb = 0.0
    for vid in videos:
        out = output_root / os.path.basename(vid)
        if out.exists() and out.stat().st_size > 0:
            continue
        tasks.append((vid, str(out)))
        total_size_gb += os.path.getsize(vid) / (1024 ** 3)

    print(f"Found {len(videos)} EPIC videos; {len(tasks)} need resizing.")
    print(f"Input size to process: {total_size_gb:.2f} GB")
    if not tasks:
        return []

    failed = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(resize_video_gpu, inp, out, short_side): inp for inp, out in tasks}
        with tqdm(total=total_size_gb, unit="GB", desc="Resizing EPIC videos") as pbar:
            for future in as_completed(futures):
                inp = futures[future]
                if not future.result():
                    failed.append(inp)
                pbar.update(os.path.getsize(inp) / (1024 ** 3))

    print(f"Resize done. Failed: {len(failed)}")
    if failed:
        print("\n".join(failed[:20]))
    return failed


if Path("/kaggle/working").exists() and PREPROCESS_EPIC:
    failed = resize_dataset(
        EPIC_INPUT_PATTERN,
        EPIC_RESIZED_ROOT,
        short_side=EPIC_SHORT_SIDE,
        num_threads=EPIC_RESIZE_THREADS,
    )
    resized_count = len(list(EPIC_RESIZED_ROOT.glob("*.MP4")))
    print("Resized EPIC root:", EPIC_RESIZED_ROOT)
    print("Resized EPIC videos:", resized_count)
    if resized_count == 0:
        raise RuntimeError("No resized EPIC videos were produced. Check EPIC_INPUT_PATTERN and ffmpeg/NVENC availability.")
else:
    print("[INFO] Skipping EPIC preprocessing; dataset link cell will use original videos if available.")


## Dataset Links

Use the old pretraining strategy: prefer resized EPIC videos from `/tmp/epic-kitchens-100-224`, then create the expected `data/VITRA_1M` symlinks after copying `VITRA` from `prepare-vitra-resources`.


In [ ]:
%%bash
set -e
mkdir -p data/VITRA_1M/Video data/VITRA_1M/Annotation

# Prefer resized EPIC videos produced by the preprocess cell. This matches the old notebook's fast eval path.
rm -rf data/VITRA_1M/Video/Epic-Kitchen_root
if [ -d /tmp/epic-kitchens-100-224 ] && compgen -G "/tmp/epic-kitchens-100-224/*.MP4" > /dev/null; then
  ln -sfn /tmp/epic-kitchens-100-224 data/VITRA_1M/Video/Epic-Kitchen_root
  echo "[INFO] Linked resized EPIC videos from /tmp/epic-kitchens-100-224"
elif compgen -G "/kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4" > /dev/null; then
  mkdir -p data/VITRA_1M/Video/Epic-Kitchen_root
  for vid in /kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4; do
    ln -sf "$vid" "data/VITRA_1M/Video/Epic-Kitchen_root/$(basename "$vid")"
  done
  echo "[WARN] Resized EPIC videos not found; linked original EPIC videos instead. Eval will be much slower."
else
  echo "[WARN] EPIC video input pattern not found. Check attached Kaggle datasets."
fi

if [ -d /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 ]; then
  ln -sfn /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 data/VITRA_1M/Video/Somethingsomething-v2_root
else
  echo "[WARN] SSV2 video root not found. EPIC-only eval can still run."
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic data/VITRA_1M/Annotation/epic
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 data/VITRA_1M/Annotation/ssv2
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/statistics ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/statistics data/VITRA_1M/Annotation/statistics
fi

find data/VITRA_1M -maxdepth 3 -type l -print | head -100


In [ ]:
CONFIG = "vitra/configs/human_pretrain.json"

WEIGHT_CANDIDATES = [
    "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt",
    "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt",
    "/kaggle/input/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt",
    "/kaggle/input/vitra-vla-3b/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt",
]

def resolve_checkpoint(candidates):
    for item in candidates:
        path = Path(item)
        if path.is_dir() and (path / "weights.pt").exists():
            return str(path / "weights.pt")
        if path.exists():
            return str(path)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = sorted(input_root.glob("**/final-epoch=0-step=16000.ckpt/weights.pt"))
        if matches:
            print("Auto-discovered 16k checkpoint:", matches[0])
            return str(matches[0])

        nearby = sorted(input_root.glob("**/final-epoch=0-step=*.ckpt/weights.pt"))[:20]
        if nearby:
            print("Found checkpoint-like files, but not step 16000:")
            for path in nearby:
                print("  ", path)

    raise FileNotFoundError(
        "Could not find the 16k VITRA checkpoint. Attach the Kaggle model input that contains "
        "final-epoch=0-step=16000.ckpt/weights.pt, or edit WEIGHT_CANDIDATES in this cell."
    )

WEIGHTS_16K = resolve_checkpoint(WEIGHT_CANDIDATES)
print("Using checkpoint:", WEIGHTS_16K)

PALIGEMMA_CANDIDATES = [
    "/kaggle/input/models/ldthanh/paligemma2-3b-mix-224/transformers/default/1",
    "/kaggle/input/paligemma2-3b-mix-224/transformers/default/1",
]

def resolve_paligemma(candidates):
    for item in candidates:
        path = Path(item)
        if (path / "config.json").exists():
            return str(path)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = sorted(input_root.glob("**/paligemma2-3b-mix-224/**/config.json"))
        if not matches:
            matches = sorted(input_root.glob("**/config.json"))
            matches = [p for p in matches if "paligemma" in str(p).lower()]
        if matches:
            print("Auto-discovered PaliGemma config:", matches[0])
            return str(matches[0].parent)

    raise FileNotFoundError(
        "Could not find local PaliGemma model config.json. Attach the Kaggle model input "
        "ldthanh/paligemma2-3b-mix-224, or edit PALIGEMMA_CANDIDATES in this cell."
    )

PALIGEMMA_PATH = resolve_paligemma(PALIGEMMA_CANDIDATES)
print("Using PaliGemma backbone:", PALIGEMMA_PATH)

config_path = Path(CONFIG)
configs_for_eval = json.loads(config_path.read_text())
configs_for_eval.setdefault("vlm", {})["pretrained_model_name_or_path"] = PALIGEMMA_PATH
train_dataset_cfg = configs_for_eval.setdefault("train_dataset", {})
train_dataset_cfg["data_root_dir"] = "data/VITRA_1M"
train_dataset_cfg["data_mix"] = "magic_mix_epic_ssv2"
train_dataset_cfg["augmentation"] = False
train_dataset_cfg["state_mask_prob"] = 0.0
train_dataset_cfg["set_none_ratio"] = 0.0
config_path.write_text(json.dumps(configs_for_eval, indent=4))
print("Using data mix:", train_dataset_cfg["data_mix"])
print("Using data root:", train_dataset_cfg["data_root_dir"])

CUTOFF = 128000
SMOKE_BATCHES = 200      # 12,800 examples with batch_size=64
FULL_BATCHES = 16400     # 1,049,600 examples, about 3h+ per setting in prior runs
NUM_WORKERS = 32

OUT_ROOT = Path("ablation_results")
OUT_ROOT.mkdir(exist_ok=True)

assert Path(CONFIG).exists(), CONFIG
assert Path(WEIGHTS_16K).exists(), WEIGHTS_16K


In [ ]:
def run_eval(name, batches, ablate_state="none", ablate_fov="none", repeated_diffusion_steps=None, phase="smoke"):
    out_dir = OUT_ROOT / phase
    out_dir.mkdir(parents=True, exist_ok=True)
    output_jsonl = out_dir / f"{name}.jsonl"
    cmd = [
        "python", "scripts/evaluate_pretrained_loss.py",
        "--config", CONFIG,
        "--weights", WEIGHTS_16K,
        "--eval_dataset", "epic",
        "--eval_sampler_step", str(CUTOFF),
        "--eval_batches", str(batches),
        "--seen_sampler_steps", str(CUTOFF),
        "--output_jsonl", str(output_jsonl),
        "--num_workers", str(NUM_WORKERS),
        "--ablate_state", ablate_state,
        "--ablate_fov", ablate_fov,
    ]
    if repeated_diffusion_steps is not None:
        cmd += ["--repeated_diffusion_steps", str(repeated_diffusion_steps)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    return output_jsonl.with_name(f"{output_jsonl.stem}.epic.summary.json")

def load_summaries(phase):
    rows = []
    for path in sorted((OUT_ROOT / phase).glob("*.summary.json")):
        data = json.loads(path.read_text())
        metrics = data["metrics"]
        rows.append({
            "file": path.name,
            "checkpoint": "16k",
            "dataset": data["dataset"],
            "examples": data["eval_examples"],
            "ablate_state": data.get("ablation", {}).get("ablate_state", "none"),
            "ablate_fov": data.get("ablation", {}).get("ablate_fov", "none"),
            "repeated_diffusion_steps": data.get("ablation", {}).get("repeated_diffusion_steps"),
            "loss": metrics["loss"]["mean"],
            "left_hand_6d": metrics["left_hand_6d"]["mean"],
            "right_hand_6d": metrics["right_hand_6d"]["mean"],
            "left_hand_joints": metrics["left_hand_joints"]["mean"],
            "right_hand_joints": metrics["right_hand_joints"]["mean"],
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["examples", "ablate_state", "ablate_fov", "repeated_diffusion_steps"])
        baseline = df[(df["ablate_state"] == "none") & (df["ablate_fov"] == "none")]
        if not baseline.empty:
            base_loss = baseline.iloc[0]["loss"]
            df["delta_vs_baseline"] = df["loss"] - base_loss
            df["relative_vs_baseline_pct"] = 100 * df["delta_vs_baseline"] / base_loss
    return df


## Smoke Ablation

Run all cheap settings on the same clean EPIC subset. These are the settings to include in the report if full eval time is limited.

In [ ]:
SMOKE_SETTINGS = [
    dict(name="baseline", ablate_state="none", ablate_fov="none"),
    dict(name="zero_state", ablate_state="zero_state", ablate_fov="none"),
    dict(name="no_state", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state", ablate_state="shuffle_state", ablate_fov="none"),
    dict(name="zero_fov", ablate_state="none", ablate_fov="zero_fov"),
    dict(name="shuffle_fov", ablate_state="none", ablate_fov="shuffle_fov"),
    dict(name="rds1", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=1),
    dict(name="rds4", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=4),
]

for setting in SMOKE_SETTINGS:
    run_eval(batches=SMOKE_BATCHES, phase="smoke", **setting)

smoke_df = load_summaries("smoke")
smoke_df.to_csv(OUT_ROOT / "smoke_summary.csv", index=False)
smoke_df

## Full Test-Split Ablation

Full eval is expensive. Turn `RUN_FULL` on only after smoke results look useful. Suggested minimum: `no_state`; suggested second setting: `shuffle_state`.

In [ ]:
RUN_FULL = False

FULL_SETTINGS = [
    dict(name="no_state_full", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state_full", ablate_state="shuffle_state", ablate_fov="none"),
    # Uncomment if smoke shows a clear FOV effect and you still have GPU time.
    # dict(name="zero_fov_full", ablate_state="none", ablate_fov="zero_fov"),
]

if RUN_FULL:
    for setting in FULL_SETTINGS:
        run_eval(batches=FULL_BATCHES, phase="full", **setting)

full_df = load_summaries("full")
if not full_df.empty:
    full_df.to_csv(OUT_ROOT / "full_summary.csv", index=False)
full_df

In [ ]:
combined = pd.concat([load_summaries("smoke"), load_summaries("full")], ignore_index=True)
if not combined.empty:
    combined.to_csv(OUT_ROOT / "ablation_summary.csv", index=False)
    print(combined.to_string(index=False))
combined

In [ ]:
%%bash
set -e
zip -qr ../vitra_ablation_results.zip ablation_results
ls -lh ../vitra_ablation_results.zip